## Speech Embeddings - PCA - Linear Mixed Model (Baseline & Interaction effects)

### Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
import statsmodels.formula.api as smf

### Data

In [ ]:
# Load and define columns
DATA_PATH ='.xlsx'
df = pd.read_excel(DATA_PATH)

NR_COL     = "Nr"
TIME_COL   = "Timepoint"
TASK_COL   = "Task"
GROUP_COL  = "Group"          
AGE_COL    = "Age"
SEX_COL    = "Sex"

EXCLUDE_NR = [] # Fill in subject ID if child needs to be exluded (i.e. pubertal voice change) 


# Speech Features (Embeddings)
wavlm_features = [f"wavlm_{i:03d}" for i in range(768)]
whisper_features = [f"whisp_{i:03d}" for i in range(768)]

# Acoustic Features (eGeMaps)
egemaps_covars = {
    "Pitch": "F0semitoneFrom27.5Hz_sma3nz_amean",
    "Spectral": "mfcc1V_sma3nz_stddevNorm",
    "Temporal": "StddevVoicedSegmentLengthSec"
}
egemaps_features = list(egemaps_covars.values())

print("Defined eGeMAPS covariates:", egemaps_features)


# Preprocessing
def preprocess_speech_embeddings(
    df: pd.DataFrame,
    feature_cols: list,
    nr_col: str,
    time_col: str,
    group_col: str,
    task_col: str,
    exclude_nr: list = None,
):
    df_clean = df.copy()
    
    # --- Convert subject ID to string ---
    df_clean[nr_col] = df_clean[nr_col].astype(str)

    # --- Exclude predefined subjects ---
    exclude_nr = [str(x) for x in exclude_nr] if exclude_nr else []
    if exclude_nr:
        before = df_clean[nr_col].nunique()
        df_clean = df_clean[~df_clean[nr_col].isin(exclude_nr)].copy()
        after = df_clean[nr_col].nunique()
        print(f"Excluded {before - after} subjects: {exclude_nr}")

    # --- Ensure stable categorical encoding (string-based) ---
    df_clean[GROUP_COL] = df_clean[GROUP_COL].astype(str)
    df_clean[GROUP_COL] = pd.Categorical(df_clean[GROUP_COL], categories=["2", "1"], ordered=True) # 2 = Controls, 1 = ADHD (controls = Reference)

    df_clean[TIME_COL] = pd.Categorical(df_clean[TIME_COL].astype(str), categories=["t0", "t2"], ordered=True) # t0 = Reference
    df_clean[SEX_COL]  = pd.Categorical(df_clean[SEX_COL].astype(str), categories=["1", "2", "3"], ordered=False) # nominal control variable
    
    # --- Create baseline age ---
    df_clean["Age_baseline"] = (
        df_clean.loc[df_clean[TIME_COL] == "t0"].groupby(NR_COL)[AGE_COL].transform("first")
    )
    df_clean["Age_baseline"] = df_clean.groupby(NR_COL)["Age_baseline"].transform("first")
    print(df_clean[["Nr", "Timepoint", "Age", "Age_baseline"]].head(11))
    
    # --- Check but do not drop rows with NaNs ---
    n_missing = df_clean[feature_cols].isna().any(axis=1).sum()
    print(f"\n {n_missing} rows contain at least one NaN in feature columns.")

    # --- Z-scale embeddings separately per task ---
    for t in df_clean["Task"].dropna().unique():
        mask = df_clean["Task"] == t

        # WavLM scaling within task
        if any(f in df_clean.columns for f in wavlm_features):
            sub = df_clean.loc[mask, wavlm_features]
            mean_wavlm = sub.mean()
            std_wavlm = sub.std(ddof=0).replace(0, np.nan)  # avoid div by 0
            df_clean.loc[mask, wavlm_features] = (sub - mean_wavlm) / std_wavlm

        # Whisper scaling within task
        if any(f in df_clean.columns for f in whisper_features):
            sub = df_clean.loc[mask, whisper_features]
            mean_whisp = sub.mean()
            std_whisp = sub.std(ddof=0).replace(0, np.nan)
            df_clean.loc[mask, whisper_features] = (sub - mean_whisp) / std_whisp
        
        # --- eGeMAPS covariates ---
        sub_ege = df_clean.loc[mask, egemaps_features]
        df_clean.loc[mask, egemaps_features] = (sub_ege - sub_ege.mean()) / sub_ege.std(ddof=0).replace(0, np.nan)
    
    print("Z-scaled embeddings and eGeMAPS features separately within each task.")
    
    # --- Rename (alias) eGeMAPS features to concise names for modeling ---
    df_clean["eGeMaps_Pitch"]     = df_clean["F0semitoneFrom27.5Hz_sma3nz_amean"]
    df_clean["eGeMaps_Spectral"]  = df_clean["mfcc1V_sma3nz_stddevNorm"]
    df_clean["eGeMaps_Temporal"]  = df_clean["StddevVoicedSegmentLengthSec"]
    
    return df_clean

# Run preprocessing
initial_n = df[NR_COL].nunique()

df_clean = preprocess_speech_embeddings(
    df=df,
    feature_cols=wavlm_features + whisper_features + egemaps_features,
    nr_col=NR_COL,
    time_col=TIME_COL,
    group_col=GROUP_COL,
    task_col=TASK_COL,
    exclude_nr=EXCLUDE_NR,
)

final_n = df_clean[NR_COL].nunique()

# --- Quick sanity checks ---
print("----- Feature Definition Summary -----")
print(f"Number of WavLM features:   {len(wavlm_features)}")
print(f"Number of Whisper features: {len(whisper_features)}")
print(f"Number of eGeMaps features: {len(egemaps_features)}")
print("----- Dataset Summary -----")
print(f"Initial subjects: {initial_n}")
print(f"Remaining subjects after exclusion and filtering: {final_n}")


### PCA

In [ ]:
# --- Choose dataset ---
DATASET = "wavlm"   # change to "whisper" or "wavlm" when needed
TOP_K = 10          # number of top loadings to keep per PC
MIN_FEATURES = 3    # minimum usable features for PCA

# --- Columns ---
NR_COL = "Nr"
GROUP_COL = "Group"
TASK_COL = "Task"

# Only use baseline (t0) data for PCA fitting
df_base = df_clean[df_clean["Timepoint"] == "t0"].copy()

# ============================================================
# Select features automatically
# ============================================================
if DATASET == "wavlm":
    feature_prefix = "wavlm_"
elif DATASET == "whisper":
    feature_prefix = "whisp_"
else:
    raise ValueError("DATASET must be 'wavlm' or 'whisper'")

embedding_features = [c for c in df_clean.columns if c.startswith(feature_prefix)]
print(f"Using {len(embedding_features)} embedding features for {DATASET.upper()}.")

# ============================================================
# Helper: PCA per task
# ============================================================
def _pca_task(task):
    """Run PCA on baseline (t0) data, project both sessions, and store PC scores for LMM."""
    # --- Fit PCA on baseline only ---
    dat_base = df_base.loc[df_base[TASK_COL] == task].copy()
    if len(embedding_features) < MIN_FEATURES:
        return None, None, None

    cols = [NR_COL, GROUP_COL] + embedding_features
    sub_base = dat_base[cols].dropna().drop_duplicates(subset=[NR_COL]).copy()
    if sub_base[GROUP_COL].nunique() < 2 or len(sub_base) < 6:
        return None, None, None

    # --- Fit PCA on baseline data ---
    X_base = sub_base[embedding_features].astype(float).to_numpy()
    pca = PCA().fit(X_base)

    # keep PCs until ≥80% variance explained
    var_cum = np.cumsum(pca.explained_variance_ratio_)
    #n_keep = int(np.searchsorted(var_cum, 0.80) + 1)
    N_PCS = 10  # fixed number of PCs across tasks
    n_keep = min(N_PCS, X_base.shape[1])
    expl = pca.explained_variance_ratio_[:n_keep]

    # --- Baseline scores for group comparison ---
    scores_base = pca.transform(X_base)[:, :n_keep]
    g = sub_base[GROUP_COL].astype(str).astype(int).to_numpy()
    rows = []
    for i in range(n_keep):
        s1, s2 = scores_base[g == 1, i], scores_base[g == 2, i]
        pv = stats.ttest_ind(s1, s2, equal_var=False, nan_policy="omit").pvalue
        rows.append({
            "task": task,
            "PC": i + 1,
            "explained_var": float(expl[i]),
            "p": float(pv)
        })
    out = pd.DataFrame(rows)
    out["q"] = multipletests(out["p"], method="fdr_bh")[1] if len(out) > 1 else np.nan

    # --- Project BOTH t0 and t2 data onto same PCA ---
    dat_full = df_clean.loc[df_clean[TASK_COL] == task].copy()
    X_full = dat_full[embedding_features].astype(float).to_numpy()
    scores_full = pca.transform(X_full)[:, :n_keep]

    # --- Store PC scores per subject/timepoint ---
    pc_df = dat_full[[NR_COL, GROUP_COL, "Timepoint", "Task"]].copy()
    for i in range(scores_full.shape[1]):
        pc_df[f"PC{i+1}"] = scores_full[:, i]

    return out, sub_base[[NR_COL, GROUP_COL]], pc_df


# ============================================================
# Helper: feature–PC correlations and top loadings
# ============================================================
def pca_feature_correlations(task):
    """Compute correlations between original features and PCs (per task)."""
    dat = df_base.loc[df_base[TASK_COL] == task].copy()
    cols = [NR_COL, GROUP_COL] + embedding_features
    sub = dat[cols].dropna().drop_duplicates(subset=[NR_COL]).copy()
    if sub[GROUP_COL].nunique() < 2 or len(sub) < 6:
        return None, None

    X = sub[embedding_features].astype(float).to_numpy()
    pca = PCA().fit(X)

    var_cum = np.cumsum(pca.explained_variance_ratio_)
    N_PCS = 10  # fixed number of PCs across tasks
    n_keep = min(N_PCS, X.shape[1])
    scores = pca.transform(X)[:, :n_keep]

    scores_df = pd.DataFrame(scores, columns=[f"PC{i + 1}" for i in range(n_keep)])
    X_df = pd.DataFrame(X, columns=embedding_features)

    rows, top_rows = [], []
    for i in range(n_keep):
        r = X_df.corrwith(scores_df.iloc[:, i]).dropna()

        for feat, ri in r.items():
            rows.append({"task": task, "feature": feat, "PC": i + 1, "corr": float(ri)})

        # top loadings per PC
        g2 = pd.DataFrame(r.abs().sort_values(ascending=False).head(TOP_K))
        g2 = g2.reset_index().rename(columns={"index": "feature", 0: "corr"})
        g2["PC"] = i + 1
        g2["task"] = task
        top_rows.append(g2)

    return pd.DataFrame(rows), pd.concat(top_rows, ignore_index=True)

# ============================================================
# Run PCA for all tasks
# ============================================================
pca_results, corr_all, top_all, pc_all = [], [], [], []
for t in sorted(df_clean[TASK_COL].unique()):
    o, _, pc_df = _pca_task(t)
    ca, ta = pca_feature_correlations(t)
    if o is not None: pca_results.append(o)
    if ca is not None: corr_all.append(ca)
    if ta is not None: top_all.append(ta)
    if pc_df is not None: pc_all.append(pc_df)

# Combine PC scores for LMM
pca_for_lmm = pd.concat(pc_all, ignore_index=True) if pc_all else pd.DataFrame()

# ============================================================
# Save all outputs (including LMM-ready PCs)
# ============================================================
out_pca_file = f"pca_results_PCA10_{DATASET}_24.11.25.xlsx"

with pd.ExcelWriter(out_pca_file, engine="xlsxwriter") as writer:
    if pca_results:
        pd.concat(pca_results).to_excel(writer, sheet_name="pca_all", index=False)
    if corr_all:
        pd.concat(corr_all).to_excel(writer, sheet_name="corr_all", index=False)
    if top_all:
        pd.concat(top_all).to_excel(writer, sheet_name="top_all", index=False)
    if not pca_for_lmm.empty:
        pca_for_lmm.to_excel(writer, sheet_name="pca_for_lmm", index=False)

print(f"\nPCA results and LMM-ready PCs saved to {out_pca_file}")
print(f"pca_for_lmm shape: {pca_for_lmm.shape}")


### LMM on PCA's

In [ ]:
# ============================================================
# LINEAR MIXED MODEL (LMM) – Speech Embeddings (PCA Components)
# ============================================================

# ---------------- CONFIG ----------------
MIN_N_PER_GROUP = 5
DATASET = "wavlm"  # change to "whisper" or "wavlm" as needed
OUTFILE = f"LMM_results_PCA_{DATASET}.xlsx"

# Columns from preprocessing
NR_COL     = "Nr"
GROUP_COL  = "Group"
TIME_COL   = "Timepoint"
TASK_COL   = "Task"
AGE_COL    = "Age_baseline"
SEX_COL    = "Sex"

# Merge demographic covariates into PCA dataset
cols_cov = ["Nr", "Age_baseline", "Sex"]
df_cov = df_clean[cols_cov].drop_duplicates(subset=["Nr"])
pca_for_lmm = pca_for_lmm.merge(df_cov, on="Nr", how="left")

print(f"After merge: pca_for_lmm shape = {pca_for_lmm.shape}")
print("Columns:", pca_for_lmm.columns.tolist())

pca_for_lmm[GROUP_COL] = pd.Categorical(pca_for_lmm[GROUP_COL], categories=["2", "1"], ordered=True)
pca_for_lmm[TIME_COL]  = pd.Categorical(pca_for_lmm[TIME_COL], categories=["t0", "t2"], ordered=True)


# ============================================================
# Main LMM analysis: One model per task × PC
# ============================================================

pca_results = []

for task in sorted(pca_for_lmm[TASK_COL].unique()):
    print(f"\n--- Task: {task} ---")
    df_task = pca_for_lmm[pca_for_lmm[TASK_COL] == task].copy()

    pc_cols = [c for c in df_task.columns if c.startswith("PC")]

    for pc in tqdm(pc_cols, desc=f"Task {task}"):

        cols_needed = [pc, GROUP_COL, TIME_COL, AGE_COL, SEX_COL, NR_COL]
        df_pc = df_task[cols_needed].dropna().copy().reset_index(drop=True)

        # Skip low-variance or small samples
        if df_pc.empty or df_pc[GROUP_COL].nunique() < 2:
            continue  # just skip if no two groups present

        df_pc["group_id"] = pd.Categorical(df_pc[NR_COL]).codes

        # --- LMM formula and terms ---
        formula = (
            f'Q("{pc}") ~ C({GROUP_COL}, Treatment(reference="2")) * '
            f'C({TIME_COL}, Treatment(reference="t0")) + {AGE_COL} + C({SEX_COL})'
        )
        term_baseline = f'C({GROUP_COL}, Treatment(reference="2"))[T.1]'
        term_interaction = (
            f'C({GROUP_COL}, Treatment(reference="2"))[T.1]:'
            f'C({TIME_COL}, Treatment(reference="t0"))[T.t2]'
        )

        # --- Fit LMM ---
        try:
            model = smf.mixedlm(formula, data=df_pc, groups=df_pc["group_id"], re_formula="1").fit(
                reml=True, method="powell", maxiter=1000, disp=False
            )

            # Extract baseline and interaction terms
            beta_base = model.params.get(term_baseline, np.nan)
            se_base = model.bse.get(term_baseline, np.nan)
            pval_base = model.pvalues.get(term_baseline, np.nan)
            ci_lower_base, ci_upper_base = np.nan, np.nan
            if term_baseline in model.params.index:
                ci = model.conf_int().loc[term_baseline]
                ci_lower_base, ci_upper_base = ci[0], ci[1]

            beta_int = model.params.get(term_interaction, np.nan)
            se_int = model.bse.get(term_interaction, np.nan)
            pval_int = model.pvalues.get(term_interaction, np.nan)
            ci_lower_int, ci_upper_int = np.nan, np.nan
            if term_interaction in model.params.index:
                ci = model.conf_int().loc[term_interaction]
                ci_lower_int, ci_upper_int = ci[0], ci[1]

            direction = (
                "ADHD ↑ vs HC" if beta_int > 0 else
                "ADHD ↓ vs HC" if beta_int < 0 else "no_diff"
            )

            pca_results.append({
                "task": task, "PC": pc,
                "coef_baseline": beta_base, "se_baseline": se_base,
                "p_val_baseline": pval_base,
                "ci_lower_baseline": ci_lower_base, "ci_upper_baseline": ci_upper_base,
                "std_effect_baseline": beta_base / se_base if se_base and se_base > 0 else np.nan,
                "coef_interaction": beta_int, "se_interaction": se_int,
                "p_val_interaction": pval_int,
                "ci_lower_interaction": ci_lower_int, "ci_upper_interaction": ci_upper_int,
                "std_effect_interaction": beta_int / se_int if se_int and se_int > 0 else np.nan,
                "direction_interaction": direction,
                "df_resid": getattr(model, "df_resid", np.nan),
                "method_used": "LMM", "skip_reason": None,
                "convergence": str(model.converged),
            })

        except Exception as e:
            print(f"LMM failed for {task}-{pc}: {e}")
            pca_results.append({
                "task": task, "PC": pc,
                "skip_reason": "model_failed", "method_used": "none"
            })


# ============================================================
# Collect results
# ============================================================

pca_df = pd.DataFrame(pca_results)

n_models = pca_df.query("method_used == 'LMM' and skip_reason.isna()").shape[0] ###new
print(f"LMMs fitted: {n_models}") ###new

# --- FDR correction per task (baseline + interaction)
for term_type, p_col, fdr_col in [
    ("baseline", "p_val_baseline", "fdr_p_baseline"),
    ("interaction", "p_val_interaction", "fdr_p_interaction"),
]:
    pca_df[fdr_col] = np.nan
    for task, sub in pca_df.groupby("task"):
        mask_valid = (sub["skip_reason"].isna()) & (sub[p_col].notna())
        if mask_valid.any():
            _, pvals_corr, _, _ = multipletests(sub.loc[mask_valid, p_col], method="fdr_bh")
            pca_df.loc[sub.index[mask_valid], fdr_col] = pvals_corr

# --- Significance flags
pca_df["p_sig_baseline"] = pca_df["p_val_baseline"] < 0.05
pca_df["q_sig_baseline"] = pca_df["fdr_p_baseline"] < 0.05
pca_df["p_sig_interaction"] = pca_df["p_val_interaction"] < 0.05
pca_df["q_sig_interaction"] = pca_df["fdr_p_interaction"] < 0.05

# --- Task summary ---
summary_list = []
for task, sub in pca_df.groupby("task"):
    abs_eff_int = np.abs(sub["std_effect_interaction"].dropna())
    abs_eff_base = np.abs(sub["std_effect_baseline"].dropna())
    summary_list.append({
        "task": task,
        "n_p_sig_baseline": sub["p_sig_baseline"].sum(),
        "n_q_sig_baseline": sub["q_sig_baseline"].sum(),
        "mean_abs_std_effect_baseline": abs_eff_base.mean() if not abs_eff_base.empty else np.nan,
        "max_abs_std_effect_baseline": abs_eff_base.max() if not abs_eff_base.empty else np.nan,
        "n_p_sig_interaction": sub["p_sig_interaction"].sum(),
        "n_q_sig_interaction": sub["q_sig_interaction"].sum(),
        "mean_abs_std_effect_interaction": abs_eff_int.mean() if not abs_eff_int.empty else np.nan,
        "max_abs_std_effect_interaction": abs_eff_int.max() if not abs_eff_int.empty else np.nan,
    })

summary_df = pd.DataFrame(summary_list)

# ============================================================
# Save to Excel
# ============================================================

with pd.ExcelWriter(OUTFILE, engine="xlsxwriter") as writer:
    pca_df.to_excel(writer, sheet_name="pc_level_results", index=False)
    summary_df.to_excel(writer, sheet_name="task_summary", index=False)

print(f"\nLMM PCA-level analysis complete. Results saved to {OUTFILE}")


### Post-Hoc Analysis (eGeMaps features as covariates)

In [ ]:
# POST-HOC LMM — Re-analyse significant PCs controlling for eGeMAPS features

# ---------------- CONFIG ----------------
DATASET = "whisper"  # "wavlm" or "whisper"
OUTFILE = f"LMM_results_PCA_{DATASET}_POSTHOC_eGeMAPS.xlsx"

# Columns
NR_COL     = "Nr"
GROUP_COL  = "Group"
TIME_COL   = "Timepoint"
TASK_COL   = "Task"
AGE_COL    = "Age_baseline"
SEX_COL    = "Sex"

# Acoustic covariates (already z-scaled and renamed)
PROSODY_FEATS = ["eGeMaps_Pitch", "eGeMaps_Spectral", "eGeMaps_Temporal"]

# ============================================================
# Identify q-significant task–PC combinations from original LMM
# ============================================================

sig_df = pca_df.loc[
    (
        (pca_df["q_sig_baseline"] == True) |
        (pca_df["q_sig_interaction"] == True) |
        (pca_df["p_sig_baseline"] == True) |
        (pca_df["p_sig_interaction"] == True)
    ),
    ["task", "PC"]
].drop_duplicates()

print(f"Found {len(sig_df)} p- or q-significant task–PC combinations for post-hoc analysis:")
print(sig_df)

# ============================================================
# Prepare the dataset for post-hoc LMMs
# ============================================================

cols_cov = ["Nr", "Task", "Timepoint", "eGeMaps_Pitch", "eGeMaps_Spectral", "eGeMaps_Temporal"]
df_cov = df_clean[cols_cov].drop_duplicates(subset=["Nr", "Task", "Timepoint"]).copy()

# Make sure key columns are consistent types
for df in [pca_for_lmm, df_cov]:
    df["Nr"] = df["Nr"].astype(str)
    df["Task"] = df["Task"].astype(str)
    df["Timepoint"] = df["Timepoint"].astype(str)

# Merge using all 3 keys
pca_for_lmm = pca_for_lmm.merge(df_cov, on=["Nr", "Task", "Timepoint"], how="left")
print(f"Clean merge complete. Shape: {pca_for_lmm.shape}")

# Quick verification
check = pca_for_lmm[["Task", "Timepoint", "eGeMaps_Pitch", "eGeMaps_Spectral", "eGeMaps_Temporal"]].groupby(["Task", "Timepoint"]).mean()
print("\nMean eGeMAPS covariates per Task/Timepoint:")
print(check.round(3).head())

print("\nPreview of pca_for_lmm:")
print(pca_for_lmm.head(10))   # shows the first 10 rows


# Set categorical ordering (important for contrasts)
pca_for_lmm[GROUP_COL] = pd.Categorical(pca_for_lmm[GROUP_COL], categories=["2", "1"], ordered=True)
pca_for_lmm[TIME_COL]  = pd.Categorical(pca_for_lmm[TIME_COL], categories=["t0", "t2"], ordered=True)


# ============================================================
# Run post-hoc LMMs for significant PCs, adding eGeMAPS covariates
# ============================================================

posthoc_results = []

for _, row in sig_df.iterrows():
    task, pc = row["task"], row["PC"]
    print(f"\n--- Re-analysing {task} - {pc} (with eGeMAPS covariates) ---")

    df_task = pca_for_lmm.loc[pca_for_lmm[TASK_COL] == task].copy()
    cols_needed = [pc, GROUP_COL, TIME_COL, AGE_COL, SEX_COL, NR_COL] + PROSODY_FEATS
    df_pc = df_task[cols_needed].dropna().copy().reset_index(drop=True)

    if df_pc.empty or df_pc[GROUP_COL].nunique() < 2:
        print(f"Skipping {task}-{pc}: insufficient data.")
        continue

    df_pc["group_id"] = pd.Categorical(df_pc[NR_COL]).codes

    # --- LMM formula (adds eGeMAPS covariates as controls)
    formula = (
        f'Q("{pc}") ~ C({GROUP_COL}, Treatment(reference="2")) * '
        f'C({TIME_COL}, Treatment(reference="t0")) + {AGE_COL} + C({SEX_COL}) + '
        f'{PROSODY_FEATS[0]} + {PROSODY_FEATS[1]} + {PROSODY_FEATS[2]}'
    )

    term_baseline = f'C({GROUP_COL}, Treatment(reference="2"))[T.1]'
    term_interaction = (
        f'C({GROUP_COL}, Treatment(reference="2"))[T.1]:'
        f'C({TIME_COL}, Treatment(reference="t0"))[T.t2]'
    )

    # --- Fit the LMM
    try:
        model = smf.mixedlm(formula, data=df_pc, groups=df_pc["group_id"], re_formula="1").fit(
            reml=True, method="powell", maxiter=1000, disp=False
        )

        beta_base = model.params.get(term_baseline, np.nan)
        se_base = model.bse.get(term_baseline, np.nan)
        pval_base = model.pvalues.get(term_baseline, np.nan)

        beta_int = model.params.get(term_interaction, np.nan)
        se_int = model.bse.get(term_interaction, np.nan)
        pval_int = model.pvalues.get(term_interaction, np.nan)

        posthoc_results.append({
            "task": task,
            "PC": pc,
            "coef_baseline": beta_base,
            "se_baseline": se_base,
            "std_effect_baseline": beta_base / se_base if se_base > 0 else np.nan,
            "p_val_baseline": pval_base,
            "coef_interaction": beta_int,
            "se_interaction": se_int,
            "std_effect_interaction": beta_int / se_int if se_int > 0 else np.nan,
            "p_val_interaction": pval_int,
            "convergence": str(model.converged),
        })

    except Exception as e:
        print(f"Post-hoc model failed for {task}-{pc}: {e}")
        posthoc_results.append({
            "task": task,
            "PC": pc,
            "skip_reason": "model_failed",
            "method_used": "LMM"
        })

# ============================================================
# FDR correction & save
# ============================================================

posthoc_df = pd.DataFrame(posthoc_results)

for term_type, p_col, fdr_col in [
    ("baseline", "p_val_baseline", "fdr_p_baseline"),
    ("interaction", "p_val_interaction", "fdr_p_interaction"),
]:
    posthoc_df[fdr_col] = np.nan
    for task, sub in posthoc_df.groupby("task"):
        mask_valid = sub[p_col].notna()
        if mask_valid.any():
            _, pvals_corr, _, _ = multipletests(sub.loc[mask_valid, p_col], method="fdr_bh")
            posthoc_df.loc[sub.index[mask_valid], fdr_col] = pvals_corr

# --- Save both sets of results
with pd.ExcelWriter(OUTFILE, engine="xlsxwriter") as writer:
    posthoc_df.to_excel(writer, sheet_name="posthoc_results", index=False)
    sig_df.to_excel(writer, sheet_name="original_significant_PCs", index=False)

print(f"\nPost-hoc LMM analysis complete. Results saved to {OUTFILE}")
